In [1]:
from collections import Counter
import pandas as pd
from sklearn.cluster import MeanShift, estimate_bandwidth
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# 读取数据
data = pd.read_excel(r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage5_deybe\296×87.xlsx")
X = data.iloc[:, 3:]
X
feature_names = X.columns
X = StandardScaler().fit_transform(X)
# 数据归一化。
min_val = np.min(X, axis=0)
max_val = np.max(X, axis=0)
X = (X - min_val) / (max_val - min_val)   
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X.shape

(296, 87)

In [2]:
# 获取已报道结构编号
def extract_numbers(file_path):
    df = pd.read_excel(file_path)
    valid_numbers = []
    for index, value in df['composition'].items():
        if not pd.isna(value):
            valid_numbers.append(int(df.at[index, 'number']))
    return valid_numbers

file_path = r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage5_deybe\benchmark5.xlsx"
reported_structures = extract_numbers(file_path)
print(reported_structures)

for j in range(30, 100, 5):
    j = j/100
    # 使用主成分分析PCA将数据降维。
    pca = PCA(n_components= j)
    pca_data = pca.fit_transform(X)
    num_pca = pca_data.shape[1]    

    for q in np.arange(0.01, 0.1, 0.001):  # bandwidth参数
        q = round(q, 2)   #         保留两位小数
        bandwidth = estimate_bandwidth(pca_data, quantile=q)
        ms = MeanShift(bandwidth=bandwidth, cluster_all=True, bin_seeding=True)
        ms.fit(pca_data)
        labels = ms.labels_
        unique_labels = np.unique(labels)
        
        if 7 > len(np.unique(labels)) > 1:
            print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")
        
            # 将标签添加到原始数据
            data['cluster_label'] = labels
            print(f"\nPCA= {j}:", f"聚类数量 k={len(np.unique(labels))}:",f"\n带宽评估指数 q= {q}:")
            dict = {}    
            # 统计每个簇的情况。
            label_1 = []
            for num  in  reported_structures:
                label = labels[num-1]
                label_1.append(int(label))
            print(label_1)

            for label in unique_labels:
                # 当前簇的所有样本
                cluster_samples = data[data['cluster_label'] == label]
                total_in_cluster = len(cluster_samples)           
                # 已报道样本数
                reported_in_cluster = len([num for num in cluster_samples['number'] 
                                            if num in reported_structures])
                print(f"簇编号 {label}: {reported_in_cluster}/{total_in_cluster} "
                        f"(已报道比例: {reported_in_cluster/total_in_cluster:.2%})")
            
            # 打印标签分布
            print("标签分布:", Counter(labels))
        else:
            pass


[14, 15, 16, 17, 95, 111, 142, 144, 153, 157, 171, 180, 199, 261, 283]

Silhouette score: 0.36 
Calinski-Harabasz score: 189.64

PCA= 0.4: 聚类数量 k=5: 
带宽评估指数 q= 0.09:
[1, 1, 1, 0, 1, 0, 0, 3, 0, 0, 1, 0, 2, 1, 1]
簇编号 0: 6/89 (已报道比例: 6.74%)
簇编号 1: 7/111 (已报道比例: 6.31%)
簇编号 2: 1/52 (已报道比例: 1.92%)
簇编号 3: 1/39 (已报道比例: 2.56%)
簇编号 4: 0/5 (已报道比例: 0.00%)
标签分布: Counter({np.int64(1): 111, np.int64(0): 89, np.int64(2): 52, np.int64(3): 39, np.int64(4): 5})

Silhouette score: 0.36 
Calinski-Harabasz score: 189.64

PCA= 0.4: 聚类数量 k=5: 
带宽评估指数 q= 0.09:
[1, 1, 1, 0, 1, 0, 0, 3, 0, 0, 1, 0, 2, 1, 1]
簇编号 0: 6/89 (已报道比例: 6.74%)
簇编号 1: 7/111 (已报道比例: 6.31%)
簇编号 2: 1/52 (已报道比例: 1.92%)
簇编号 3: 1/39 (已报道比例: 2.56%)
簇编号 4: 0/5 (已报道比例: 0.00%)
标签分布: Counter({np.int64(1): 111, np.int64(0): 89, np.int64(2): 52, np.int64(3): 39, np.int64(4): 5})

Silhouette score: 0.36 
Calinski-Harabasz score: 189.64

PCA= 0.4: 聚类数量 k=5: 
带宽评估指数 q= 0.09:
[1, 1, 1, 0, 1, 0, 0, 3, 0, 0, 1, 0, 2, 1, 1]
簇编号 0: 6/89 (已报道比例: 6.74%)
簇编号 1: 

In [3]:
pca = PCA(n_components= 0.4, random_state=42)
new_data = pca.fit_transform(X)
print(new_data.shape)
ratios = pca.explained_variance_ratio_   # 表示每个主成分所解释的方差占原始数据集方差的比例。
components = pca.components_    # 主成分在原始特征上的权重系数。
X = pd.DataFrame(X) 
# 将权重系数矩阵转换为 DataFrame，用于将权重系数矩阵输出到 Excel 中，便于后续分析特征重要性。
components_df = pd.DataFrame(
    components,
    columns=X.columns,  # 使用原始数据的列名作为权重系数的列名
    index=[f"PC{i+1}" for i in range(components.shape[0])]  # 使用 PC1, PC2 作为行名
)
# 输出到 Excel 文件
output_file = "pca_components.xlsx"
components_df.to_excel(output_file, index=True)

bandwidth = estimate_bandwidth(new_data, quantile=0.09)
ms = MeanShift(bandwidth=bandwidth, cluster_all=True, bin_seeding=True)
ms.fit(new_data)
labels = ms.labels_
unique_labels = np.unique(labels)

df = pd.DataFrame(labels)
df.to_excel("Meanshift_result_1.xlsx", index=False)

if len(np.unique(labels)) > 1:
    print(f"\nSilhouette score: {silhouette_score(new_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(new_data, labels):.2f}")
else:
    pass
# 将标签添加到原始数据
data['cluster_label'] = labels
print(f"聚类数量 k={len(np.unique(labels))}:", )
dict = {}    
# 统计每个簇的情况。
label_1 = []
for num  in  reported_structures:
    label = labels[num-1]
    label_1.append(int(label))
print(label_1)

for label in unique_labels:
    # 当前簇的所有样本
    cluster_samples = data[data['cluster_label'] == label]
    total_in_cluster = len(cluster_samples)           
    # 已报道样本数
    reported_in_cluster = len([num for num in cluster_samples['number'] 
                                     if num in reported_structures])
    print(f"簇编号 {label}: {reported_in_cluster}/{total_in_cluster} "
                  f"(已报道比例: {reported_in_cluster/total_in_cluster:.2%})")
        
    # 打印标签分布
print("标签分布:",  Counter(labels))

(296, 3)

Silhouette score: 0.36 
Calinski-Harabasz score: 189.64
聚类数量 k=5:
[1, 1, 1, 0, 1, 0, 0, 3, 0, 0, 1, 0, 2, 1, 1]
簇编号 0: 6/89 (已报道比例: 6.74%)
簇编号 1: 7/111 (已报道比例: 6.31%)
簇编号 2: 1/52 (已报道比例: 1.92%)
簇编号 3: 1/39 (已报道比例: 2.56%)
簇编号 4: 0/5 (已报道比例: 0.00%)
标签分布: Counter({np.int64(1): 111, np.int64(0): 89, np.int64(2): 52, np.int64(3): 39, np.int64(4): 5})


In [4]:
import pandas as pd
from sklearn.decomposition import PCA
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import Workbook

# 创建一个新的 Excel 工作簿
wb = Workbook()
ws = wb.active

# 将 DataFrame 写入 Excel 工作表
for r in dataframe_to_rows(components_df, index=True, header=True):
    ws.append(r)

# 设置红色加粗字体
red_bold_font = Font(color="FF0000", bold=True)

# 对每个主成分，标记权重系数绝对值最大的前 10 个数
for row in range(2, ws.max_row + 1):  # 从第 2 行开始（跳过表头）
    # 获取当前行的权重系数
    values = [ws.cell(row=row, column=col).value for col in range(2, ws.max_column + 1)]
    
    # 检查并替换 None 值为 0
    values = [0 if v is None else v for v in values]
    
    # 计算绝对值并排序，找到前 10 个最大值的索引
    abs_values = [abs(v) for v in values]
    top_10_indices = sorted(range(len(abs_values)), key=lambda i: abs_values[i], reverse=True)[:10]
    
    # 对前 10 个最大值的单元格设置红色加粗字体
    for idx in top_10_indices:
        ws.cell(row=row, column=idx + 2).font = red_bold_font  # +2 是因为列从第 2 列开始

# 保存 Excel 文件
output_file = "pca_components_highlighted.xlsx"
wb.save(output_file)

print(f"权重系数矩阵已保存到 {output_file}，并标记了绝对值最大的前 10 个数。")

权重系数矩阵已保存到 pca_components_highlighted.xlsx，并标记了绝对值最大的前 10 个数。
